# Walmart Weekly Sales: Experiments

This notebook tests three improvements on top of `Notebook.ipynb`, **without touching the deployed model**:

1. **A validation split that contains Thanksgiving and Christmas.** The original split (train 2010–2011 / validate 2012) validates on a period that stops on 2012-10-26, so the two biggest peaks of the year were never tested.
2. **Lag features** (sales of the same store/department one year earlier, store/department historical level).
3. **Retraining the final model on all available data (2010 → 2012)** before predicting the Kaggle test set.

All models are evaluated with the official Kaggle metric, **WMAE** (holiday weeks weigh 5×).

In [1]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import plotly.graph_objects as go

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OrdinalEncoder
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor

## Data loading & base features (same as `Notebook.ipynb`)

In [2]:
train = pd.read_csv("./Data/train.csv")
test = pd.read_csv("./Data/test.csv")
features = pd.read_csv("./Data/features.csv")
stores = pd.read_csv("./Data/stores.csv")

feature_store_merged = features.merge(stores, how="inner", on="Store")

THANKSGIVING_WEEK = {2010: "2010-11-26", 2011: "2011-11-25", 2012: "2012-11-23", 2013: "2013-11-29"}
CHRISTMAS_WEEK    = {2010: "2010-12-31", 2011: "2011-12-30", 2012: "2012-12-28", 2013: "2013-12-27"}
SUPER_BOWL_WEEK   = {2010: "2010-02-12", 2011: "2011-02-11", 2012: "2012-02-10", 2013: "2013-02-08"}
LABOR_DAY_WEEK    = {2010: "2010-09-10", 2011: "2011-09-09", 2012: "2012-09-07", 2013: "2013-09-06"}
HOLIDAYS = {"Thanksgiving": THANKSGIVING_WEEK, "Christmas": CHRISTMAS_WEEK,
            "SuperBowl": SUPER_BOWL_WEEK, "LaborDay": LABOR_DAY_WEEK}

MARKDOWN_COLS = ["MarkDown1", "MarkDown2", "MarkDown3", "MarkDown4", "MarkDown5"]


def build_base_features(df):
    df = df.merge(feature_store_merged, how="inner", on=["Store", "Date", "IsHoliday"])
    df["Date"] = pd.to_datetime(df["Date"])
    df["Year"] = df["Date"].dt.year
    df["Week_Of_Year"] = df["Date"].dt.isocalendar().week.astype(int)
    for name, dates in HOLIDAYS.items():
        ref = df["Date"].dt.year.map({y: pd.Timestamp(d) for y, d in dates.items()})
        df[f"Days_to_{name}"] = (ref - df["Date"]).dt.days
    df[MARKDOWN_COLS] = df[MARKDOWN_COLS].fillna(0)
    df["IsHoliday"] = df["IsHoliday"].astype(int)
    return df.sort_values(["Store", "Dept", "Date"]).reset_index(drop=True)


train_df = build_base_features(train)
test_df = build_base_features(test)

BASE_FEATURES = ["Store", "Dept", "IsHoliday", "Temperature", "Fuel_Price", *MARKDOWN_COLS,
                 "CPI", "Unemployment", "Type", "Size", "Year", "Week_Of_Year",
                 "Days_to_Thanksgiving", "Days_to_Christmas", "Days_to_SuperBowl", "Days_to_LaborDay"]

print("train:", train_df["Date"].min().date(), "->", train_df["Date"].max().date(), train_df.shape)
print("test: ", test_df["Date"].min().date(), "->", test_df["Date"].max().date(), test_df.shape)

train: 2010-02-05 -> 2012-10-26 (421570, 22)
test:  2012-11-02 -> 2013-07-26 (115064, 21)


## Evaluation helpers

* `wmae`: the official Kaggle metric.
* `MAE Thanks/Xmas`: error on Thanksgiving and Christmas weeks only, the weeks the original validation could not measure.

In [3]:
def wmae(y_true, y_pred, is_holiday):
    w = np.where(is_holiday, 5, 1)
    return np.sum(w * np.abs(np.asarray(y_true) - np.asarray(y_pred))) / np.sum(w)


def peak_mask(dates):
    peaks = [pd.Timestamp(d) for d in list(THANKSGIVING_WEEK.values()) + list(CHRISTMAS_WEEK.values())]
    return dates.isin(peaks).values


def evaluate(name, y_true, y_pred, val_df):
    peaks = peak_mask(val_df["Date"])
    return {
        "Experiment": name,
        "WMAE": wmae(y_true, y_pred, val_df["IsHoliday"] == 1),
        "MAE": mean_absolute_error(y_true, y_pred),
        "R²": r2_score(y_true, y_pred),
        "MAE Thanks/Xmas": mean_absolute_error(y_true[peaks], y_pred[peaks]) if peaks.any() else np.nan,
    }


def make_models():
    # Same hyper-parameters as the tuned models of Notebook.ipynb; XGBoost & LightGBM on GPU
    return {
        "Random Forest": RandomForestRegressor(n_estimators=100, max_depth=20, min_samples_leaf=10,
                                               random_state=42, n_jobs=-1),
        "XGBoost": XGBRegressor(n_estimators=300, learning_rate=0.1, max_depth=8, subsample=0.8,
                                colsample_bytree=0.8, min_child_weight=5, random_state=42,
                                device="cuda", tree_method="hist"),
        "LightGBM": LGBMRegressor(n_estimators=500, learning_rate=0.05, max_depth=8, num_leaves=63,
                                  subsample=0.8, colsample_bytree=0.8, min_child_samples=20,
                                  random_state=42, device="gpu", verbose=-1),
    }


def make_pipeline(model, feature_cols):
    # Tree models are scale-invariant: no StandardScaler needed. Missing lags (NaN) are handled natively.
    numeric = [c for c in feature_cols if c != "Type"]
    pre = ColumnTransformer([
        ("type", OrdinalEncoder(categories=[["C", "B", "A"]]), ["Type"]),
        ("num", "passthrough", numeric),
    ])
    return Pipeline([("Preprocessing", pre), ("Model", model)])


def run_experiment(label, tr, va, feature_cols, results, predictions):
    for name, model in make_models().items():
        pipe = make_pipeline(model, feature_cols).fit(tr[feature_cols], tr["Weekly_Sales"])
        pred = pipe.predict(va[feature_cols])
        predictions[(label, name)] = pred
        results.append({"Model": name, **evaluate(label, va["Weekly_Sales"].values, pred, va)})
        print(f"done: {label} / {name}")

## 1. A validation split that contains the holiday peaks

| Split | Train | Validation | Thanksgiving / Christmas in validation? |
|---|---|---|---|
| Original | 2010-02-05 → 2011-12-30 | 2012-01-06 → 2012-10-26 | ❌ no |
| **New** | 2010-02-05 → 2011-10-28 | **2011-11-04 → 2012-10-26 (52 weeks)** | ✅ yes |

The new validation window is **exactly one year**, so it covers every season once, including Thanksgiving and Christmas 2011.

In [4]:
OLD_CUTOFF = pd.Timestamp("2011-12-30")
NEW_CUTOFF = pd.Timestamp("2011-10-28")

splits = {
    "Old split": (train_df[train_df["Date"] <= OLD_CUTOFF], train_df[train_df["Date"] > OLD_CUTOFF]),
    "New split": (train_df[train_df["Date"] <= NEW_CUTOFF], train_df[train_df["Date"] > NEW_CUTOFF]),
}

for name, (tr, va) in splits.items():
    holidays_in_val = sorted(va.loc[va["IsHoliday"] == 1, "Date"].dt.date.unique())
    print(f"{name}: train {len(tr):,} rows | val {len(va):,} rows | holiday weeks in val: {holidays_in_val}")

Old split: train 294,132 rows | val 127,438 rows | holiday weeks in val: [datetime.date(2012, 2, 10), datetime.date(2012, 9, 7)]
New split: train 267,184 rows | val 154,386 rows | holiday weeks in val: [datetime.date(2011, 11, 25), datetime.date(2011, 12, 30), datetime.date(2012, 2, 10), datetime.date(2012, 9, 7)]


In [5]:
results, predictions = [], {}

for split_name, (tr, va) in splits.items():
    run_experiment(f"{split_name} / base features", tr, va, BASE_FEATURES, results, predictions)

pd.DataFrame(results).round({"WMAE": 0, "MAE": 0, "R²": 4, "MAE Thanks/Xmas": 0})

done: Old split / base features / Random Forest


done: Old split / base features / XGBoost


done: Old split / base features / LightGBM


done: New split / base features / Random Forest


done: New split / base features / XGBoost


done: New split / base features / LightGBM


,Model,Experiment,WMAE,MAE,R²,MAE Thanks/Xmas
0,Random Forest,Old split / base features,2364.0,2330.0,0.9459,NaN
1,XGBoost,Old split / base features,2752.0,2732.0,0.9523,NaN
2,LightGBM,Old split / base features,3122.0,3094.0,0.9417,NaN
3,Random Forest,New split / base features,2779.0,2595.0,0.9304,4070.0
4,XGBoost,New split / base features,2919.0,2809.0,0.9476,3747.0
5,LightGBM,New split / base features,3498.0,3348.0,0.9318,4602.0


**Reading:** the old split's scores were optimistic. Once Thanksgiving and Christmas are in the validation window, the WMAE goes up for every model: the holiday weeks are both the hardest to predict and weighted 5×. The **new split is the honest benchmark** for the rest of this notebook.

## 2. Lag features

### Which lags are allowed?

A lag is only legitimate if its value is **known at prediction time**. The Kaggle test set covers 2012-11-02 → 2013-07-26, i.e. up to **39 weeks** after the last known sales. A `lag_1` (last week's sales) is therefore **not available** for most test weeks: using it would require recursive forecasting and would leak in validation.

Lags of **52 weeks or more** are always known, both for the test set and for our 52-week validation window. We build:

| Feature | Description |
|---|---|
| `Sales_Lag_52` | Sales of the same store/department exactly 52 weeks (364 days) earlier: same weekday, same holiday week |
| `Sales_Lag_52_Mean4` | Mean of lags 52 to 55 weeks: a smoother "last year around this time" level |
| `Store_Dept_Mean` | Average weekly sales of the store/department over the **training period only** |
| `Store_Dept_Median` | Same with the median, robust to holiday spikes |

364 days = 52 weeks exactly, so Thanksgiving 2011 (2011-11-25) is aligned with Thanksgiving 2010 (2010-11-26), and Christmas weeks are aligned too.

In [6]:
LAG_FEATURES = ["Sales_Lag_52", "Sales_Lag_52_Mean4", "Store_Dept_Mean", "Store_Dept_Median"]


def add_lag_features(df, history, stats_source):
    """
    df:           rows to enrich
    history:      known sales (Store, Dept, Date, Weekly_Sales) used for the 52+ week lags
    stats_source: rows used to compute the store/department level (training period only)
    """
    df = df.copy()
    hist = history[["Store", "Dept", "Date", "Weekly_Sales"]]
    for k in [52, 53, 54, 55]:
        shifted = hist.assign(Date=hist["Date"] + pd.Timedelta(weeks=k)).rename(columns={"Weekly_Sales": f"lag_{k}"})
        df = df.merge(shifted, on=["Store", "Dept", "Date"], how="left")
    df["Sales_Lag_52"] = df["lag_52"]
    df["Sales_Lag_52_Mean4"] = df[[f"lag_{k}" for k in [52, 53, 54, 55]]].mean(axis=1)
    df = df.drop(columns=[f"lag_{k}" for k in [52, 53, 54, 55]])

    stats = (stats_source.groupby(["Store", "Dept"])["Weekly_Sales"]
             .agg(Store_Dept_Mean="mean", Store_Dept_Median="median").reset_index())
    return df.merge(stats, on=["Store", "Dept"], how="left")


tr, va = splits["New split"]
# history = training period only: guarantees that no validation sales leak into the features
tr_lag = add_lag_features(tr, history=tr, stats_source=tr)
va_lag = add_lag_features(va, history=tr, stats_source=tr)

print("Share of validation rows with a known 52-week lag:", f"{va_lag['Sales_Lag_52'].notna().mean():.1%}")
print("Share of training rows with a known 52-week lag:  ", f"{tr_lag['Sales_Lag_52'].notna().mean():.1%}",
      "(the first year has no history)")

Share of validation rows with a known 52-week lag: 97.1%
Share of training rows with a known 52-week lag:   41.8% (the first year has no history)


In [7]:
run_experiment("New split / base + lag features", tr_lag, va_lag, BASE_FEATURES + LAG_FEATURES, results, predictions)

comparison = (pd.DataFrame(results)
              .sort_values(["Experiment", "WMAE"])
              .reset_index(drop=True))
comparison.round({"WMAE": 0, "MAE": 0, "R²": 4, "MAE Thanks/Xmas": 0})

done: New split / base + lag features / Random Forest


done: New split / base + lag features / XGBoost


done: New split / base + lag features / LightGBM


,Model,Experiment,WMAE,MAE,R²,MAE Thanks/Xmas
0,LightGBM,New split / base + lag features,2070.0,1961.0,0.9637,2985.0
1,Random Forest,New split / base + lag features,2293.0,2017.0,0.9451,4425.0
2,XGBoost,New split / base + lag features,2645.0,2443.0,0.8124,4137.0
3,Random Forest,New split / base features,2779.0,2595.0,0.9304,4070.0
4,XGBoost,New split / base features,2919.0,2809.0,0.9476,3747.0
5,LightGBM,New split / base features,3498.0,3348.0,0.9318,4602.0
6,Random Forest,Old split / base features,2364.0,2330.0,0.9459,NaN
7,XGBoost,Old split / base features,2752.0,2732.0,0.9523,NaN
8,LightGBM,Old split / base features,3122.0,3094.0,0.9417,NaN


In [8]:
new_split = comparison[comparison["Experiment"].str.startswith("New split")]
gain = new_split.pivot(index="Model", columns="Experiment", values="WMAE")
# explicit column order (pivot sorts columns alphabetically)
gain = gain[["New split / base features", "New split / base + lag features"]]
gain.columns = ["WMAE base", "WMAE + lags"]
gain["Gain"] = gain["WMAE base"] - gain["WMAE + lags"]
gain["Gain %"] = gain["Gain"] / gain["WMAE base"] * 100
gain.sort_values("WMAE + lags").round(1)

,WMAE base,WMAE + lags,Gain,Gain %
Model,,,,
LightGBM,3498.5,2069.7,1428.8,40.8
Random Forest,2778.6,2293.3,485.4,17.5
XGBoost,2918.7,2644.5,274.2,9.4


### Actual vs predicted on the new validation window (with Thanksgiving & Christmas)

In [9]:
best_lag_model = gain["WMAE + lags"].idxmin()
plot_df = va[["Date"]].copy()
plot_df["Actual"] = va["Weekly_Sales"].values
plot_df["Base features"] = predictions[("New split / base features", best_lag_model)]
plot_df["Base + lag features"] = predictions[("New split / base + lag features", best_lag_model)]
weekly = plot_df.groupby("Date", as_index=False).sum()

fig = go.Figure()
fig.add_trace(go.Scatter(x=weekly["Date"], y=weekly["Actual"], name="Actual", line=dict(width=3, color="black")))
fig.add_trace(go.Scatter(x=weekly["Date"], y=weekly["Base features"], name=f"{best_lag_model}: base features",
                         line=dict(width=2, dash="dot")))
fig.add_trace(go.Scatter(x=weekly["Date"], y=weekly["Base + lag features"], name=f"{best_lag_model}: + lag features",
                         line=dict(width=2, dash="dash")))
fig.update_layout(title=f"Total weekly sales, Nov 2011 → Oct 2012: actual vs {best_lag_model}",
                  title_x=0.5, xaxis_title="Week", yaxis_title="Total Weekly Sales ($)",
                  template="plotly_white", hovermode="x unified", width=1100, height=500)
fig.show()

In [10]:
fitted = (make_pipeline(make_models()[best_lag_model], BASE_FEATURES + LAG_FEATURES)
          .fit(tr_lag[BASE_FEATURES + LAG_FEATURES], tr_lag["Weekly_Sales"])
          .named_steps["Model"])
# LightGBM reports split counts by default: use gain to be comparable with the other models
raw_importance = (fitted.booster_.feature_importance(importance_type="gain")
                  if hasattr(fitted, "booster_") else fitted.feature_importances_)
importances = pd.Series(raw_importance,
                        index=["Type"] + [c for c in BASE_FEATURES + LAG_FEATURES if c != "Type"]
                        ).sort_values(ascending=False)
print(f"{best_lag_model}: share of total gain per feature")
(importances / importances.sum()).head(10).round(3)

LightGBM: share of total gain per feature


Store_Dept_Mean         0.711
Store_Dept_Median       0.155
Dept                    0.042
Sales_Lag_52_Mean4      0.029
Sales_Lag_52            0.025
Size                    0.007
Week_Of_Year            0.005
Days_to_Christmas       0.005
Days_to_Thanksgiving    0.004
Days_to_SuperBowl       0.002
dtype: float64

## 3. Retrain on all data before predicting the test set

In `Notebook.ipynb`, the model used for the submission was trained on **2010–2011 only**: the 43 most recent weeks (Jan → Oct 2012), the closest to the test period, were never used for training.

For the final submission, once the model and features are chosen on the validation split, the standard practice is to **retrain on everything we know** (2010-02-05 → 2012-10-26). This cannot be scored without submitting to Kaggle, so we check it visually: the forecast must continue the historical pattern smoothly.

In [11]:
final_model_name = best_lag_model
final_features = BASE_FEATURES + LAG_FEATURES

# history and store/department statistics now use ALL known sales
train_full = add_lag_features(train_df, history=train_df, stats_source=train_df)
test_full = add_lag_features(test_df, history=train_df, stats_source=train_df)
print("Share of test rows with a known 52-week lag:", f"{test_full['Sales_Lag_52'].notna().mean():.1%}")

final_pipeline = make_pipeline(make_models()[final_model_name], final_features)
final_pipeline.fit(train_full[final_features], train_full["Weekly_Sales"])
test_full["Weekly_Sales"] = final_pipeline.predict(test_full[final_features])

Share of test rows with a known 52-week lag: 98.2%


In [12]:
submission = pd.DataFrame({
    "Id": test_full["Store"].astype(str) + "_" + test_full["Dept"].astype(str) + "_"
          + test_full["Date"].dt.strftime("%Y-%m-%d"),
    "Weekly_Sales": test_full["Weekly_Sales"],
})
sample = pd.read_csv("./Data/sampleSubmission.csv")
assert len(submission) == len(sample) and set(submission["Id"]) == set(sample["Id"])

submission.to_csv("walmart_submission_v2.csv", index=False)
print(f"walmart_submission_v2.csv saved ({final_model_name} + lag features, trained on 2010-2012). Not submitted.")
submission.head()

walmart_submission_v2.csv saved (LightGBM + lag features, trained on 2010-2012). Not submitted.


,Id,Weekly_Sales
0,1_1_2012-11-02,39990.928976
1,1_1_2012-11-09,20891.685316
2,1_1_2012-11-16,19859.046441
3,1_1_2012-11-23,21500.720207
4,1_1_2012-11-30,25321.359839


In [13]:
history_weekly = train_df.groupby("Date", as_index=False)["Weekly_Sales"].sum()
forecast_weekly = test_full.groupby("Date", as_index=False)["Weekly_Sales"].sum()

fig = go.Figure()
fig.add_trace(go.Scatter(x=history_weekly["Date"], y=history_weekly["Weekly_Sales"], name="Known sales",
                         line=dict(width=2, color="black")))
fig.add_trace(go.Scatter(x=forecast_weekly["Date"], y=forecast_weekly["Weekly_Sales"],
                         name=f"Forecast ({final_model_name} + lags, trained on all data)",
                         line=dict(width=2, color="crimson")))
fig.update_layout(title="Total weekly sales: history and Kaggle test-period forecast", title_x=0.5,
                  xaxis_title="Week", yaxis_title="Total Weekly Sales ($)",
                  template="plotly_white", hovermode="x unified", width=1100, height=500)
fig.show()

## 4. Out-of-fold target encoding

`Store_Dept_Mean` and `Store_Dept_Median` are **target encodings**: the average sales of each store/department pair. On the validation rows they are computed from the training period only, so the validation score is clean.

On the **training rows**, however, each row's own sales are part of the mean it receives: a small in-sample leak (each row weighs ~1/90 of its pair's mean). The model may learn to trust these features a bit more than it should.

**Out-of-fold (OOF) encoding** removes this leak: the training weeks are split into 5 folds, and the statistics of each row are computed **from the other 4 folds only**. No training row ever sees its own target. The folds are made of whole **weeks**, so a row never gets information from its own week either.

In [14]:
from sklearn.model_selection import KFold

STAT_FEATURES = ["Store_Dept_Mean", "Store_Dept_Median"]


def oof_store_dept_stats(df, n_splits=5, seed=42):
    """Store/department mean & median for each row, computed without the row's own week."""
    df = df.copy()
    weeks = np.sort(df["Date"].unique())
    out = pd.DataFrame(index=df.index, columns=STAT_FEATURES, dtype=float)
    for fit_idx, enc_idx in KFold(n_splits=n_splits, shuffle=True, random_state=seed).split(weeks):
        fit_rows = df["Date"].isin(weeks[fit_idx])
        stats = (df[fit_rows].groupby(["Store", "Dept"])["Weekly_Sales"]
                 .agg(Store_Dept_Mean="mean", Store_Dept_Median="median"))
        enc = df.loc[~fit_rows, ["Store", "Dept"]].join(stats, on=["Store", "Dept"])
        out.loc[enc.index, STAT_FEATURES] = enc[STAT_FEATURES].values
    df[STAT_FEATURES] = out
    return df


tr_oof = oof_store_dept_stats(tr_lag)   # lags unchanged, only the two statistics are re-computed

print("Correlation between in-sample and OOF Store_Dept_Mean on training rows:",
      f"{tr_lag['Store_Dept_Mean'].corr(tr_oof['Store_Dept_Mean']):.4f}")
print("Mean absolute difference: $" f"{(tr_lag['Store_Dept_Mean'] - tr_oof['Store_Dept_Mean']).abs().mean():,.0f}")
print("Training rows without an OOF value (pair seen in a single fold):", f"{tr_oof['Store_Dept_Mean'].isna().mean():.2%}")

Correlation between in-sample and OOF Store_Dept_Mean on training rows: 0.9999
Mean absolute difference: $162
Training rows without an OOF value (pair seen in a single fold): 0.02%


In [15]:
oof_results = []
for encoding, train_frame in [("in-sample", tr_lag), ("out-of-fold", tr_oof)]:
    for name, model in make_models().items():
        pipe = make_pipeline(model, BASE_FEATURES + LAG_FEATURES).fit(
            train_frame[BASE_FEATURES + LAG_FEATURES], train_frame["Weekly_Sales"])
        val_pred = pipe.predict(va_lag[BASE_FEATURES + LAG_FEATURES])
        train_pred = pipe.predict(train_frame[BASE_FEATURES + LAG_FEATURES])
        row = {"Model": name, **evaluate(f"lags, {encoding} encoding", va_lag["Weekly_Sales"].values, val_pred, va_lag)}
        row["Train WMAE"] = wmae(train_frame["Weekly_Sales"], train_pred, train_frame["IsHoliday"] == 1)
        row["Val/Train WMAE ratio"] = row["WMAE"] / row["Train WMAE"]
        oof_results.append(row)
        print(f"done: {encoding} / {name}")

oof_table = pd.DataFrame(oof_results).sort_values(["Model", "Experiment"]).reset_index(drop=True)
oof_table.round({"WMAE": 0, "MAE": 0, "R²": 4, "MAE Thanks/Xmas": 0, "Train WMAE": 0, "Val/Train WMAE ratio": 2})

done: in-sample / Random Forest


done: in-sample / XGBoost


done: in-sample / LightGBM


done: out-of-fold / Random Forest


done: out-of-fold / XGBoost


done: out-of-fold / LightGBM


,Model,Experiment,WMAE,MAE,R²,MAE Thanks/Xmas,Train WMAE,Val/Train WMAE ratio
0,LightGBM,"lags, in-sample encoding",2070.0,1961.0,0.9636,2990.0,1582.0,1.31
1,LightGBM,"lags, out-of-fold encoding",2086.0,1970.0,0.9637,3064.0,1629.0,1.28
2,Random Forest,"lags, in-sample encoding",2293.0,2017.0,0.9451,4425.0,1632.0,1.41
3,Random Forest,"lags, out-of-fold encoding",2364.0,2066.0,0.9422,4668.0,1700.0,1.39
4,XGBoost,"lags, in-sample encoding",2645.0,2443.0,0.8124,4137.0,1276.0,2.07
5,XGBoost,"lags, out-of-fold encoding",2061.0,1944.0,0.9649,3077.0,1312.0,1.57


In [16]:
lgbm_importance = {}
for encoding, train_frame in [("in-sample", tr_lag), ("out-of-fold", tr_oof)]:
    fitted = (make_pipeline(make_models()["LightGBM"], BASE_FEATURES + LAG_FEATURES)
              .fit(train_frame[BASE_FEATURES + LAG_FEATURES], train_frame["Weekly_Sales"])
              .named_steps["Model"])
    gain = pd.Series(fitted.booster_.feature_importance(importance_type="gain"),
                     index=["Type"] + [c for c in BASE_FEATURES + LAG_FEATURES if c != "Type"])
    lgbm_importance[encoding] = gain / gain.sum()

pd.DataFrame(lgbm_importance).sort_values("in-sample", ascending=False).head(8).round(3)

,in-sample,out-of-fold
Store_Dept_Mean,0.711,0.682
Store_Dept_Median,0.155,0.181
Dept,0.042,0.042
Sales_Lag_52_Mean4,0.029,0.030
Sales_Lag_52,0.025,0.025
Size,0.007,0.007
Week_Of_Year,0.005,0.005
Days_to_Christmas,0.005,0.006


**Reading:**

| Model | WMAE in-sample → OOF | R² in-sample → OOF | Val/Train WMAE ratio in-sample → OOF |
|---|---|---|---|
| **XGBoost** | 2,645 → **2,061** (-22%) | 0.812 → **0.965** | 2.07 → 1.57 |
| LightGBM | 2,070 → 2,086 (+1%) | 0.964 → 0.964 | 1.31 → 1.28 |
| Random Forest | 2,293 → 2,364 (+3%) | 0.945 → 0.942 | 1.41 → 1.39 |

* **The in-sample leak was real, and it explains XGBoost's R² drop.** With in-sample encoding, XGBoost had by far the largest gap between validation and training error (ratio 2.07): it over-trusted `Store_Dept_Mean` and produced a few very large errors. With OOF encoding its R² goes back to 0.965 and its WMAE drops by 22%.
* **LightGBM and Random Forest were not affected**: their scores barely move (+1% and +3%), well within what a different random seed could change. Their stronger regularisation (`min_child_samples`, `min_samples_leaf`) already prevented them from over-fitting the encoding.
* The OOF and in-sample encodings are almost identical (correlation 0.9999, mean difference $162): the leak is small, but XGBoost's deep trees (`max_depth=8`, `min_child_weight=5`) were able to exploit it.
* **XGBoost + OOF (2,061) and LightGBM + in-sample (2,070) are now tied**: a 0.4% difference is not significant. LightGBM stays better on Thanksgiving/Christmas weeks (MAE 2,990 vs 3,077).

**Decision:** the deployed LightGBM model is kept as is. OOF encoding is the methodologically safer choice and should be used for any model that is sensitive to it (XGBoost here), but it does not improve LightGBM.

## Conclusions

### 1. Validation split: the old scores were optimistic

| Model | WMAE old split (no Thanksgiving/Christmas) | WMAE new split (with them) |
|---|---|---|
| Random Forest | 2,364 | 2,779 (+18%) |
| XGBoost | 2,752 | 2,919 (+6%) |
| LightGBM | 3,124 | 3,501 (+12%) |

Every model gets worse once the holiday peaks are in the validation window. The new split is the honest benchmark.

### 2. Lag features: the biggest improvement of the project

| Model | WMAE base | WMAE + lags | Gain | MAE on Thanksgiving/Christmas weeks |
|---|---|---|---|---|
| **LightGBM** | 3,501 | **2,070** | **-41%** | 4,605 → **2,989** |
| Random Forest | 2,779 | 2,293 | -17% | 4,070 → 4,425 |
| XGBoost | 2,919 | 2,645 | -9% | 3,747 → 4,137 |

* **LightGBM + lags is the new best model** (WMAE 2,070, R² 0.9637), and the only one that also improves on the Thanksgiving/Christmas weeks.
* The gain comes mostly from **`Store_Dept_Mean` / `Store_Dept_Median`** (~87% of LightGBM's total gain): knowing the usual sales level of each store/department pair is far more informative than `Store` and `Dept` as raw IDs. The 52-week lags add the "same week last year" signal (~5%).
* Only lags of **52 weeks or more** were used, because the Kaggle test set goes up to 39 weeks ahead: shorter lags would not be available at prediction time.
* ⚠️ XGBoost with lags improves its WMAE but its **R² drops from 0.948 to 0.812**: a few very large errors appear. Section 4 shows why: XGBoost over-fits the in-sample target encoding, and out-of-fold encoding fixes it (R² 0.965, WMAE 2,061).

### 3. Retraining on all data

The final LightGBM + lags model was retrained on **2010-02-05 → 2012-10-26** and saved to `walmart_submission_v2.csv` (**not submitted**). It cannot be scored without Kaggle, but a plausibility check against the same season one year earlier is consistent:

| Total weekly sales (M$) | Thanksgiving week | Mean Nov → Jul |
|---|---|---|
| Actual 2011–12 | 66.6 | 48.6 |
| v1 submission (XGBoost, trained 2010–2011) | 64.9 | 46.2 |
| **v2 submission (LightGBM + lags, trained 2010–2012)** | **68.4** | **49.5** |

Sales were stable to slightly growing year-over-year (+2.5% on comparable weeks in 2012), and v2 follows that level while v1, which never saw 2012, sits below it.

### Next steps
* ✅ Done: LightGBM + lag features is now the final model of `Notebook.ipynb` and the model deployed in the app (feature engineering shared through `walmart_features.py`).
* Tune LightGBM's hyper-parameters on the new split (they were copied from the original notebook).
* ✅ XGBoost's R² drop with lag features is explained by the in-sample target encoding leak: see section 4, fixed by out-of-fold encoding.